In [9]:
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import torch.nn.functional as F


In [16]:
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        # transforms.Normalize((0.1307,), (0.3081,)),
    ]
)

data = datasets.MNIST(
    root='data',
    train=True,
    download=True,
    transform=transform
)
data[0][0].size()

test_data = datasets.MNIST(
    root='data',
    train=False,
    download=True,
    transform=transform
)
test_data


Dataset MNIST
    Number of datapoints: 10000
    Root location: data
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
           )

In [11]:
train_loader = DataLoader(data, 64, True)
test_loader = DataLoader(test_data, 64, shuffle=False)


In [18]:
class BaseCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=3, stride=1)
        self.conv2 = nn.Conv2d(6, 16, 3, 1)
        self.layer1 = nn.Linear(5 * 5 * 16, 128)
        self.output = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2, 2)
        x = x.view(-1, 5 * 5 * 16)
        x = F.relu(self.layer1(x))
        return F.log_softmax(self.output(x), dim=1)


class DropoutCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=3, stride=1)
        self.conv2 = nn.Conv2d(6, 16, 3, 1)
        self.layer1 = nn.Linear(5 * 5 * 16, 128)
        self.dropout = nn.Dropout(p=0.3)
        self.output = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2, 2)
        x = x.view(-1, 5 * 5 * 16)
        x = F.relu(self.layer1(x))
        x = self.dropout(x)
        return F.log_softmax(self.output(x), dim=1)


class BatchNormCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=3, stride=1)
        self.bn1 = nn.BatchNorm2d(6)
        self.conv2 = nn.Conv2d(6, 16, 3, 1)
        self.bn2 = nn.BatchNorm2d(16)
        self.layer1 = nn.Linear(5 * 5 * 16, 128)
        self.bn3 = nn.BatchNorm1d(128)
        self.output = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.max_pool2d(x, 2, 2)
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool2d(x, 2, 2)
        x = x.view(-1, 5 * 5 * 16)
        x = F.relu(self.bn3(self.layer1(x)))
        return F.log_softmax(self.output(x), dim=1)


class WeightDecayCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=3, stride=1)
        self.conv2 = nn.Conv2d(6, 16, 3, 1)
        self.layer1 = nn.Linear(5 * 5 * 16, 128)
        self.output = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2, 2)
        x = x.view(-1, 5 * 5 * 16)
        x = F.relu(self.layer1(x))
        return F.log_softmax(self.output(x), dim=1)


In [20]:
device = "cuda" if torch.cuda.is_available() else "cpu"
loss_fn = nn.CrossEntropyLoss()

def train_and_eval(model_cls, epochs=10, weight_decay=0.0):
    model = model_cls().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=weight_decay)

    for epoch in range(epochs):
        losses = 0
        model.train()
        for image, labels in train_loader:
            model.zero_grad()
            image = image.to(device)
            labels = labels.to(device)

            output = model(image)
            loss = loss_fn(output, labels)
            loss.backward()
            optimizer.step()

            losses += loss.item()
        print(f"[{model_cls.__name__}] Epoch {epoch+1}, Loss: {losses:.4f}")

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"{model_cls.__name__} Accuracy: {accuracy:.2f}% , Wrond preds : {total-correct}")
    return accuracy


In [14]:
baseline_acc = train_and_eval(BaseCNN)
dropout_acc = train_and_eval(DropoutCNN)
batchnorm_acc = train_and_eval(BatchNormCNN)

print("\nSummary")
print(f"Normal:    {baseline_acc:.2f}%")
print(f"Dropout:   {dropout_acc:.2f}%")
print(f"BatchNorm: {batchnorm_acc:.2f}%")


[BaseCNN] Epoch 1, Loss: 304.9394
[BaseCNN] Epoch 2, Loss: 87.4831
[BaseCNN] Epoch 3, Loss: 61.6455
[BaseCNN] Epoch 4, Loss: 50.5877
[BaseCNN] Epoch 5, Loss: 41.4599
[BaseCNN] Epoch 6, Loss: 35.7447
[BaseCNN] Epoch 7, Loss: 30.6507
[BaseCNN] Epoch 8, Loss: 28.0274
[BaseCNN] Epoch 9, Loss: 24.1105
[BaseCNN] Epoch 10, Loss: 20.7032
BaseCNN Accuracy: 98.65% , Wrond preds : 135
[DropoutCNN] Epoch 1, Loss: 310.1177
[DropoutCNN] Epoch 2, Loss: 102.0052
[DropoutCNN] Epoch 3, Loss: 77.7336
[DropoutCNN] Epoch 4, Loss: 62.8061
[DropoutCNN] Epoch 5, Loss: 57.6438
[DropoutCNN] Epoch 6, Loss: 49.5504
[DropoutCNN] Epoch 7, Loss: 43.2430
[DropoutCNN] Epoch 8, Loss: 39.1579
[DropoutCNN] Epoch 9, Loss: 37.2169
[DropoutCNN] Epoch 10, Loss: 33.3268
DropoutCNN Accuracy: 98.98% , Wrond preds : 102
[BatchNormCNN] Epoch 1, Loss: 145.2584
[BatchNormCNN] Epoch 2, Loss: 47.4602
[BatchNormCNN] Epoch 3, Loss: 33.4353
[BatchNormCNN] Epoch 4, Loss: 25.9716
[BatchNormCNN] Epoch 5, Loss: 21.0688
[BatchNormCNN] Epoch 

In [21]:
weight_decay_acc = train_and_eval(WeightDecayCNN, weight_decay=1e-4)

print("\nUpdated Summary")
print(f"Normal:      {baseline_acc:.2f}%")
print(f"Dropout:     {dropout_acc:.2f}%")
print(f"BatchNorm:   {batchnorm_acc:.2f}%")
print(f"WeightDecay: {weight_decay_acc:.2f}%")


[WeightDecayCNN] Epoch 1, Loss: 295.1889
[WeightDecayCNN] Epoch 2, Loss: 82.5293
[WeightDecayCNN] Epoch 3, Loss: 57.2691
[WeightDecayCNN] Epoch 4, Loss: 46.4896
[WeightDecayCNN] Epoch 5, Loss: 37.8119
[WeightDecayCNN] Epoch 6, Loss: 32.9891
[WeightDecayCNN] Epoch 7, Loss: 28.0008
[WeightDecayCNN] Epoch 8, Loss: 24.8265
[WeightDecayCNN] Epoch 9, Loss: 22.3832
[WeightDecayCNN] Epoch 10, Loss: 20.1035
WeightDecayCNN Accuracy: 98.73% , Wrond preds : 127

Updated Summary
Normal:      98.65%
Dropout:     98.98%
BatchNorm:   98.83%
WeightDecay: 98.73%


In [25]:
small_data = torch.utils.data.Subset(data, range(200))
small_loader = DataLoader(small_data, 64, True)

def train_overfit(model_cls, epochs=100):
    model = model_cls().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        losses = 0
        model.train()
        for image, labels in small_loader:
            model.zero_grad()
            image = image.to(device)
            labels = labels.to(device)

            output = model(image)
            loss = loss_fn(output, labels)
            loss.backward()
            optimizer.step()

            losses += loss.item()
        print(f"[Overfit] Epoch {epoch+1}, Loss: {losses:.4f}")

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Overfit Accuracy: {accuracy:.2f}% , Wrond preds : {total-correct}")
    return accuracy

overfitmodel = train_overfit(BaseCNN)
print(f"overfit:      {overfitmodel:.2f}%")
print(f"Normal:      {baseline_acc:.2f}%")

[Overfit] Epoch 1, Loss: 9.1388
[Overfit] Epoch 2, Loss: 9.0078
[Overfit] Epoch 3, Loss: 8.7957
[Overfit] Epoch 4, Loss: 8.5115
[Overfit] Epoch 5, Loss: 8.1253
[Overfit] Epoch 6, Loss: 8.0205
[Overfit] Epoch 7, Loss: 7.2890
[Overfit] Epoch 8, Loss: 6.6681
[Overfit] Epoch 9, Loss: 6.1161
[Overfit] Epoch 10, Loss: 5.2646
[Overfit] Epoch 11, Loss: 4.8320
[Overfit] Epoch 12, Loss: 4.3024
[Overfit] Epoch 13, Loss: 3.5920
[Overfit] Epoch 14, Loss: 3.2403
[Overfit] Epoch 15, Loss: 2.9366
[Overfit] Epoch 16, Loss: 2.7282
[Overfit] Epoch 17, Loss: 2.3419
[Overfit] Epoch 18, Loss: 1.8566
[Overfit] Epoch 19, Loss: 2.0269
[Overfit] Epoch 20, Loss: 1.7391
[Overfit] Epoch 21, Loss: 1.5290
[Overfit] Epoch 22, Loss: 1.6197
[Overfit] Epoch 23, Loss: 2.1871
[Overfit] Epoch 24, Loss: 1.2351
[Overfit] Epoch 25, Loss: 1.2079
[Overfit] Epoch 26, Loss: 1.0444
[Overfit] Epoch 27, Loss: 1.0315
[Overfit] Epoch 28, Loss: 1.0099
[Overfit] Epoch 29, Loss: 0.9021
[Overfit] Epoch 30, Loss: 1.1954
[Overfit] Epoch 31,

# Data Augmentation Notes (PyTorch)

## What is Data Augmentation?

Data augmentation creates different versions of existing training images so the model learns general features instead of memorizing exact pixels.

### Examples

```python
transforms.RandomHorizontalFlip()
transforms.RandomCrop(32, padding=4)
transforms.RandomRotation(10)
```

## Why Use Augmentation?

Without augmentation, the model may learn:

> "This exact pixel arrangement = cat"

With augmentation, the model must learn:

> "Ears + eyes + whiskers + fur = cat"

### Benefits

* Less overfitting
* Better generalization
* Higher validation/test accuracy

---

## Does Augmentation Add New Data?

### Online Augmentation (Most Common)

No new files are created.

Dataset:

```text
dog.jpg
cat.jpg
car.jpg
```

remains:

```text
dog.jpg
cat.jpg
car.jpg
```

Transforms are applied every time an image is loaded.

```python
train_dataset = CIFAR10(
    root='./data',
    train=True,
    transform=train_transform
)
```

### Offline Augmentation

Actually creates new images:

```text
dog.jpg
dog_flip.jpg
dog_crop.jpg
dog_rotate.jpg
```

Dataset size increases.

---

## How Does the Image Change Every Epoch If I Applied the Transform Only Once?

Transforms are **stored**, not executed when you create the dataset.

```python
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4)
])
```

PyTorch applies them whenever:

```python
train_dataset[idx]
```

is called.

Conceptually:

```python
def __getitem__(self, idx):

    image = load_image(idx)

    image = transform(image)

    return image
```

Since `RandomCrop` and `RandomFlip` are random, each call may produce a different image.

### Example

Same image:

```python
img, label = train_dataset[0]
```

```text
Epoch 1 -> Original Dog
Epoch 2 -> Flipped Dog
Epoch 3 -> Cropped Dog
```

Same image index, different version.

---

## Common CIFAR-10 Augmentations

```python
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),

    transforms.ToTensor(),

    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)
    )
])
```

### Test Data (No Augmentation)

```python
test_transform = transforms.Compose([
    transforms.ToTensor(),

    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)
    )
])
```

---

## Why Does Augmentation Reduce Overfitting?

Without augmentation:

```text
50,000 images
    ↓
Same 50,000 images every epoch
    ↓
Easy to memorize
```

With augmentation:

```text
50,000 images
    ↓
Different versions every epoch
    ↓
Harder to memorize
    ↓
Must learn actual features
```

---

## What Augmentations Are Safe?

Usually safe:

```python
RandomHorizontalFlip()
RandomCrop()
ColorJitter()
RandomRotation(10)
```

Example:

```text
Dog -> Flipped Dog
Dog -> Cropped Dog
```

Still a dog.

---

## What Augmentations Can Be Dangerous?

Example:

```python
RandomRotation(180)
```

For MNIST:

```text
6 -> 9
```

Label meaning may change.

### Rule

> Augmentations should change appearance while preserving the class label.
